# **Single_Day**

In [ ]:
# Single Day

import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
import os

file_path = "G:/MATLAB & Julia/NetCDF/Upscaled/analysed_sst_2021_2025_NBoB.nc"
ds = xr.open_dataset(file_path)

save_dir = "G:/MATLAB & Julia/Plot"
os.makedirs(save_dir, exist_ok=True)

parameter = ds['analysed_sst']
Param = "analysed_sst"
Unit = "C"

# If depth exists, select one
if "depth" in parameter.dims:
    print("Depth dimension detected:", ds["depth"].values)

    # Choose depth (change manually as needed)
    selected_depth = float(ds["depth"].values[0])
    print("Using depth level:", selected_depth)

    parameter = parameter.sel(depth=selected_depth, method="nearest")
else:
    print("No depth dimension found.")

selected_date = "2023-03-12"
parameter_day = parameter.sel(time=selected_date, method="nearest")
save_path = os.path.join(save_dir, f"{Param}_{selected_date}_NBoB.png")
plt.figure(figsize=(28, 16))
ax = plt.axes(projection=ccrs.PlateCarree())

# colors = [
#     (0.00, "darkblue"),
#     (0.15, "blue"),
#     (0.35, "cyan"),
#     (0.50, "green"),
#     (0.65, "yellow"),
#     (0.80, "orange"),
#     (1.00, "red")
# ]

# custom_cmap = LinearSegmentedColormap.from_list("blue_green_red", colors, N=256)

# Plot the data
im = parameter_day.plot.pcolormesh(
      ax=ax,
      cmap="jet", # give custom map if want
      transform=ccrs.PlateCarree(),
      cbar_kwargs={'label': f'{Param} ({Unit})',
          'orientation': 'vertical',
          'shrink': 0.8,
          'pad': 0.02,
  }
)

cbar = im.colorbar
cbar.ax.tick_params(labelsize=14)
cbar.ax.yaxis.label.set_size(18)
cbar.set_label(f"{Param} ({Unit})", fontsize=18)

# Cartopy
land = cfeature.NaturalEarthFeature("physical", "land", "10m")
coastline = cfeature.NaturalEarthFeature("physical", "coastline", "10m", facecolor="none")
borders = cfeature.NaturalEarthFeature("cultural", "admin_0_boundary_lines_land", "10m", facecolor="none")
lakes = cfeature.NaturalEarthFeature("physical", "lakes", "10m")
rivers = cfeature.NaturalEarthFeature("physical", "rivers_lake_centerlines", "10m", facecolor="none")

ax.add_feature(land, facecolor='lightgray', edgecolor='black', linewidth=0.5, zorder=1)
ax.add_feature(borders, edgecolor='black', linewidth=0.8, zorder=2)
ax.add_feature(coastline, edgecolor='black', linewidth=1.0, zorder=3)
ax.add_feature(rivers, edgecolor='blue', linewidth=0.8, zorder=3)
ax.add_feature(lakes, facecolor='lightblue', edgecolor='blue', linewidth=0.5, zorder=2)

# Add base features
# ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
# ax.add_feature(cfeature.COASTLINE, linewidth=1.2, zorder=2)
# ax.add_feature(cfeature.BORDERS, linestyle="--", linewidth=0.8, zorder=3)
# ax.add_feature(cfeature.LAKES, alpha=0.8, facecolor='lightblue', zorder=2)
# ax.add_feature(cfeature.RIVERS, linewidth=0.5, zorder=2)

# Optional: Add subdivision borders (e.g., states/provinces)
try:
    admin1 = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_1_states_provinces_lines',
        scale='10m',
        facecolor='none'
    )
    ax.add_feature(admin1, edgecolor='gray', linewidth=0.5, zorder=3)
except:
    print("Subdivision borders not available for this region.")

# Add gridlines with labels
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 16}
gl.ylabel_style = {'size': 16}

# Title and layout
plt.title(f"{Param} concentration on {selected_date}", fontsize=24, pad=15, weight='bold')
plt.tight_layout()
plt.savefig(save_path, dpi=300, bbox_inches="tight")
print(f"✅ Saved figure to: {save_path}")
plt.show()

# **Combined_Multi_Day_Plot**

In [ ]:
# Combine

import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os

# -------------------- Paths --------------------
file_path = r"G:/MATLAB & Julia/NetCDF/analysed_sst_1991_2020_merged.nc"
save_dir  = r"G:/MATLAB & Julia/Plot"
os.makedirs(save_dir, exist_ok=True)

# -------------------- Read dataset --------------------
ds = xr.open_dataset(file_path)

Param = "analysed_sst"
Unit  = "°C"  # change if needed (some SST products are in Kelvin)
parameter = ds[Param] - 273.15

# -------------------- Optional: handle depth --------------------
if "depth" in parameter.dims:
    print("Depth dimension detected:", ds["depth"].values)
    selected_depth = float(ds["depth"].values[0])  # choose another index if you want
    print("Using depth level:", selected_depth)
    parameter = parameter.sel(depth=selected_depth, method="nearest")
else:
    print("No depth dimension found.")

# -------------------- Make ONE map for the whole file (1991–2020) --------------------
# If your file already contains only 1991–2020, the slice is optional, but it's safe:
parameter_full = parameter.sel(time=slice("1991-01-01", "2020-12-31"))

# Reduce time dimension -> single 2D field (lat/lon)
parameter_map = parameter_full.mean(dim="time", skipna=True)

# Time range strings for title/filename
t0 = str(parameter_full["time"].min().dt.strftime("%Y-%m-%d").values)
t1 = str(parameter_full["time"].max().dt.strftime("%Y-%m-%d").values)

save_path = os.path.join(save_dir, f"{Param}_mean_{t0}_to_{t1}.png")

# -------------------- Plot --------------------
plt.figure(figsize=(28, 16))
ax = plt.axes(projection=ccrs.PlateCarree())

im = parameter_map.plot.pcolormesh(
    ax=ax,
    cmap="turbo",
    transform=ccrs.PlateCarree(),
    cbar_kwargs={
        "label": f"{Param} ({Unit})",
        "orientation": "vertical",
        "shrink": 0.8,
        "pad": 0.02,
    },
)

# Colorbar formatting
cbar = im.colorbar
cbar.ax.tick_params(labelsize=14)
cbar.ax.yaxis.label.set_size(18)
cbar.set_label(f"{Param} ({Unit})", fontsize=18)

# -------------------- Cartopy features --------------------
land = cfeature.NaturalEarthFeature("physical", "land", "10m")
coastline = cfeature.NaturalEarthFeature("physical", "coastline", "10m", facecolor="none")
borders = cfeature.NaturalEarthFeature("cultural", "admin_0_boundary_lines_land", "10m", facecolor="none")
lakes = cfeature.NaturalEarthFeature("physical", "lakes", "10m")
rivers = cfeature.NaturalEarthFeature("physical", "rivers_lake_centerlines", "10m", facecolor="none")

ax.add_feature(land, facecolor="lightgray", edgecolor="black", linewidth=0.5, zorder=1)
ax.add_feature(borders, edgecolor="black", linewidth=0.8, zorder=2)
ax.add_feature(coastline, edgecolor="black", linewidth=1.0, zorder=3)
ax.add_feature(rivers, edgecolor="blue", linewidth=0.8, zorder=3)
ax.add_feature(lakes, facecolor="lightblue", edgecolor="blue", linewidth=0.5, zorder=2)

# Optional: admin-1 boundaries
try:
    admin1 = cfeature.NaturalEarthFeature(
        category="cultural",
        name="admin_1_states_provinces_lines",
        scale="10m",
        facecolor="none",
    )
    ax.add_feature(admin1, edgecolor="gray", linewidth=0.5, zorder=3)
except Exception as e:
    print("Subdivision borders not available:", e)

# Gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5, linestyle="--")
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {"size": 16}
gl.ylabel_style = {"size": 16}

# Title / Save
plt.title(f"{Param} mean ({t0} to {t1})", fontsize=24, pad=15, weight="bold")
plt.tight_layout()
plt.savefig(save_path, dpi=300, bbox_inches="tight")
print(f"✅ Saved figure to: {save_path}")
plt.show()

# **Wind**

In [ ]:
# WIND STREAMPLOT WITH FILLED CONTOURS
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.feature as cfeature
import cmocean
from datetime import datetime, timedelta
import matplotlib.patches as mpatches
from shapely.geometry import MultiPolygon, Polygon
import os

# STEP 1: USER INPUT
nc_file = "File_LocationLocation/.nc"  # merged wind file
start_date = "2023-11-09"
days = 15
save_path = "Location/.png"

# STEP 2: LOAD DATASET
ds = xr.open_dataset(nc_file, chunks={"time": 500})
ds = ds.sel(time=~ds.get_index("time").duplicated())

u = ds["eastward_wind"]
v = ds["northward_wind"]
lon = ds["longitude"]
lat = ds["latitude"]

# Ensure lon/lat are 1D (needed for streamplot)
lon_1d = lon.values
lat_1d = lat.values
if lon_1d.ndim > 1:
    lon_1d = lon_1d[0, :]
if lat_1d.ndim > 1:
    lat_1d = lat_1d[:, 0]

# Compute wind speed
speed = np.sqrt(u**2 + v**2)

# STEP 3: DATE LIST 
date_list = [datetime.strptime(start_date, "%Y-%m-%d") + timedelta(days=i) for i in range(days)]

# STEP 4: FIGURE SETUP
nrows, ncols = 5, 3
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(32, 48), sharex=True, sharey=True)
axes = axes.flatten()
plt.subplots_adjust(wspace=0, hspace=0)

# STEP 5: BOUNDS
lon_min, lon_max = float(lon_1d.min()), float(lon_1d.max())
lat_min, lat_max = float(lat_1d.min()), float(lat_1d.max())

# Preload coastlines/land/borders
land = cfeature.NaturalEarthFeature("physical", "land", "10m")
borders = cfeature.NaturalEarthFeature("cultural", "admin_0_boundary_lines_land", "50m")

# Compute global min/max wind speed for consistent color scaling
start_dt = datetime.strptime(start_date, "%Y-%m-%d")
end_dt = start_dt + timedelta(days=days)
ds_sel = ds.sel(time=slice(start_dt, end_dt))
speed_sel = np.sqrt(ds_sel["eastward_wind"]**2 + ds_sel["northward_wind"]**2)
spd_min = float(speed_sel.min())
spd_max = float(speed_sel.max())

# STEP 6: PLOT LOOP
plot = None
for i, date in enumerate(date_list):
    try:
        u_day = u.sel(time=np.datetime64(date), method="nearest")
        v_day = v.sel(time=np.datetime64(date), method="nearest")
        spd_day = np.sqrt(u_day**2 + v_day**2)

        ax = axes[i]

        # --- Land polygons ---
        for geom in land.geometries():
            if isinstance(geom, Polygon):
                ax.add_patch(mpatches.Polygon(list(geom.exterior.coords),
                                              facecolor="lightgray",
                                              edgecolor="black",
                                              linewidth=1,
                                              zorder=100))
            elif isinstance(geom, MultiPolygon):
                for poly in geom.geoms:
                    ax.add_patch(mpatches.Polygon(list(poly.exterior.coords),
                                                  facecolor="lightgray",
                                                  edgecolor="black",
                                                  linewidth=1,
                                                  zorder=100))

        # --- Borders ---
        for geom in borders.geometries():
            try:
                x, y = geom.xy
                ax.plot(x, y, color="black", linewidth=1, zorder=101)
            except Exception:
                for line in geom.geoms:
                    x, y = line.xy
                    ax.plot(x, y, color="black", linewidth=1, zorder=101)

        # --- Filled contour (wind speed) ---
        plot = ax.contourf(lon_1d, lat_1d, spd_day, levels=200,
                           cmap="jet", vmin=spd_min, vmax=spd_max)

        # --- Contours (lines) ---
        #cs = ax.contour(lon_1d, lat_1d, spd_day, levels=15, colors='k', linewidths=0.6)
        #ax.clabel(cs, inline=True, fontsize=16, fmt="%.1f")

        # --- Streamlines (wind direction) ---
        step = 3
        ax.streamplot(lon_1d[::step], lat_1d[::step],
                      u_day.values[::step, ::step],
                      v_day.values[::step, ::step],
                      color="black", density=1.5, linewidth=0.8, arrowsize=1.5)

        # --- Geographic bounds ---
        ax.set_xlim(lon_min, lon_max)
        ax.set_ylim(lat_min, lat_max)
        ax.set_aspect('equal', adjustable='box')

        # --- Label tweaks ---
        if i % ncols != 0:
            ax.set_ylabel("")
        if i < (nrows - 1) * ncols:
            ax.set_xlabel("")
        ax.tick_params(axis='both', which='major', labelsize=16)

        # --- Date annotation ---
        ax.text(0.02, 0.95, date.strftime('%Y-%m-%d'),
                transform=ax.transAxes, fontsize=20, color='white',
                fontweight='bold', verticalalignment='top', horizontalalignment='left',
                bbox=dict(facecolor='black', alpha=0.5, boxstyle='round,pad=0.3'),
                zorder=200)

    except Exception as e:
        print(f"Error for {date}: {e}")
        axes[i].set_visible(False)

# STEP 7: COLORBAR
if plot is not None:
    cbar = fig.colorbar(plot, ax=axes, orientation='vertical', shrink=0.6, pad=0.02)
    cbar.set_label("Wind Speed (m/s)", fontsize=32, labelpad=12)
    cbar.ax.tick_params(labelsize=24)

# STEP 8: LABELS & TITLE
fig.text(0.45, 0.085, 'Longitude', ha='center', fontsize=40)
fig.text(0.09, 0.5, 'Latitude', va='center', rotation='vertical', fontsize=40)
plt.suptitle("Wind Speed & direction during Midhili", fontsize=44, y=0.90, x=0.5)

# STEP 9: SAVE AND SHOW
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved figure to: {save_path}")